# FP16 Qwen2.5-3B on a Colab T4

This notebook generates the FP16 baseline for Day 6 i.e. throughput and VRAM footprint. FP16 won't fit on my 6 GB laptop, so I run it here on the T4 (16 GB) and compare against my local 4-bit numbers.

In [1]:
# !pip install vllm

In [2]:
!pip install torchaudio --index-url https://download.pytorch.org/whl/cu130

Looking in indexes: https://download.pytorch.org/whl/cu130


In [3]:
import time, json, torch
from vllm import LLM, SamplingParams
from day1_baseline_benchmark import PROMPTS

### Note: the stdout patch below

vLLM tries to grab `sys.stdout.fileno()` to silence a subprocess, but Colab's stdout is a fake IPython stream with no file descriptor, so it crashes with `io.UnsupportedOperation: fileno`. The patch gives stdout/stderr a real `fileno` so the engine starts.

In [7]:
import sys
import io

# Patch sys.stdout and sys.stderr fileno methods if they do not exist or throw UnsupportedOperation
# to prevent vLLM initialization crash in Jupyter Notebook environments.
if not hasattr(sys.stdout, 'fileno') or isinstance(sys.stdout, io.IOBase):
    try:
        sys.stdout.fileno()
    except (io.UnsupportedOperation, AttributeError):
        sys.stdout.fileno = lambda: 1

if not hasattr(sys.stderr, 'fileno') or isinstance(sys.stderr, io.IOBase):
    try:
        sys.stderr.fileno()
    except (io.UnsupportedOperation, AttributeError):
        sys.stderr.fileno = lambda: 2

llm = LLM(model="Qwen/Qwen2.5-3B-Instruct", dtype="float16", max_model_len=2048)
sp  = SamplingParams(max_tokens=128, temperature=0.0)

INFO 09-15 10:13:55 [api_utils.py:286] non-default args: {'dtype': 'float16', 'max_model_len': 2048, 'disable_log_stats': True, 'model': 'Qwen/Qwen2.5-3B-Instruct'}
INFO 09-15 10:13:56 [model.py:684] Resolved architecture: Qwen2ForCausalLM
WARNING 09-15 10:13:56 [model.py:2355] Casting torch.bfloat16 to torch.float16.
INFO 09-15 10:13:56 [model.py:2021] Using max model len 2048
INFO 09-15 10:13:56 [kernel.py:369] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])
(EngineCore pid=8228) INFO 09-15 10:14:01 [core.py:123] Initializing a V1 LLM engine (v0.29.0) with config: model='Qwen/Qwen2.5-3B-Instruct', speculative_config=None, tokenizer='Qwen/Qwen2.5-3B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=main, tokenizer_revision=main, trust_remote_code=False, dtype=torch.float16, max_seq_len=2048, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_

Loading safetensors checkpoint shards:   0% Completed | 0/2 [00:00<?, ?it/s]


(EngineCore pid=8228) INFO 09-15 10:17:54 [default_loader.py:430] Loading weights took 26.05 seconds
(EngineCore pid=8228) INFO 09-15 10:17:55 [model_runner.py:404] Model loading took 5.79 GiB memory and 227.909138 seconds
(EngineCore pid=8228) WARNING 09-15 10:17:55 [topk_topp_sampler.py:69] FlashInfer top-p/top-k sampling unavailable: unsupported compute capability 7.5; falling back. Set VLLM_USE_FLASHINFER_SAMPLER=0 to silence.
(EngineCore pid=8228) INFO 09-15 10:17:55 [utils.py:306] Using LBNHC KV cache layout.
(EngineCore pid=8228) INFO 09-15 10:18:10 [backends.py:1094] Using cache directory: /root/.cache/vllm/torch_compile_cache/7940dda1a0/rank_0_0/backbone for vLLM's torch.compile
(EngineCore pid=8228) INFO 09-15 10:18:10 [backends.py:1155] Dynamo bytecode transform time: 11.03 s


(EngineCore pid=8228) [rank0]:W0915 10:18:18.106000 8228 torch/_inductor/utils.py:1953] Not enough SMs to use max_autotune_gemm mode


(EngineCore pid=8228) INFO 09-15 10:18:21 [backends.py:393] Compiling a graph for compile range (1, 8192) takes 11.37 s
(EngineCore pid=8228) INFO 09-15 10:18:26 [backends.py:920] collected artifacts: 37 entries, 3 artifacts, 4642180 bytes total
(EngineCore pid=8228) INFO 09-15 10:18:26 [decorators.py:719] saved AOT compiled function to /root/.cache/vllm/torch_compile_cache/torch_aot_compile/1e15346ebb6dfcd0c69192a0392f42c51db8f32b6431c1a45905e16fee649804/rank_0_0/model
(EngineCore pid=8228) INFO 09-15 10:18:26 [monitor.py:53] torch.compile took 27.46 s in total
(EngineCore pid=8228) INFO 09-15 10:18:26 [monitor.py:81] Initial profiling/warmup run took 0.21 s


Capturing CUDA graphs (FULL): 100%|██████████| 2/2 [00:00<00:00,  7.09it/s]


(EngineCore pid=8228) INFO 09-15 10:18:43 [model_runner.py:960] Graph capturing finished in 12 secs, took 0.36 GiB
(EngineCore pid=8228) INFO 09-15 10:18:44 [gpu_worker.py:625] Available KV cache memory: 5.69 GiB
(EngineCore pid=8228) INFO 09-15 10:18:44 [gpu_worker.py:640] CUDA graph memory profiling is enabled (default since v0.21.0). The current --gpu-memory-utilization=0.9200 is equivalent to --gpu-memory-utilization=0.8917 without CUDA graph memory profiling. To maintain the same effective KV cache size as before, increase --gpu-memory-utilization to 0.9483. To disable, set VLLM_MEMORY_PROFILER_ESTIMATE_CUDAGRAPHS=0.
(EngineCore pid=8228) INFO 09-15 10:18:44 [kv_cache_utils.py:2032] GPU KV cache size: 165,824 tokens, Maximum concurrency for 2,048 tokens per request: 80.97x
(EngineCore pid=8228) INFO 09-15 10:18:44 [kernel_warmup.py:124] JIT kernel warmup starting.
(EngineCore pid=8228) INFO 09-15 10:18:45 [kernel_warmup.py:134] JIT kernel warmup finished in 0.00s.


Capturing CUDA graphs (FULL): 100%|██████████| 35/35 [00:02<00:00, 12.19it/s]


(EngineCore pid=8228) INFO 09-15 10:18:55 [model_runner.py:960] Graph capturing finished in 9 secs, took 0.22 GiB
(EngineCore pid=8228) INFO 09-15 10:18:55 [gpu_worker.py:797] CUDA graph pool memory: 0.22 GiB (actual), 0.41 GiB (estimated), difference: 0.19 GiB (88.4%).
(EngineCore pid=8228) INFO 09-15 10:18:55 [gpu_worker.py:860] Free memory on device (14.46/14.56 GiB) on startup. Desired GPU memory utilization is (0.92, 13.4 GiB). Actual usage is 6.66 GiB for consumed memory (weights + non-torch), 1.05 GiB for peak activation, and 0.22 GiB for CUDAGraph memory. Replace gpu_memory_utilization config with `--kv-cache-memory=5720775824` (5.33 GiB) to fit into requested memory, or `--kv-cache-memory=6861773312` (6.39 GiB) to fully utilize gpu memory. Current kv cache memory in use is 5.69 GiB.
(EngineCore pid=8228) INFO 09-15 10:19:06 [jit_monitor.py:85] Kernel JIT monitor activated; monitored JIT compilations during inference will use mode=warn.
(EngineCore pid=8228) INFO 09-15 10:19:07

In [8]:
llm.generate(["warm up"], sp)                    # discard warm-up
torch.cuda.reset_peak_memory_stats()
t0 = time.perf_counter()
outs = llm.generate(PROMPTS, sp)
dt = time.perf_counter() - t0

Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 1/1 [00:04<00:00,  4.36s/it, est. speed input: 0.46 toks/s, output: 29.38 toks/s]


Rendering prompts:   0%|          | 0/20 [00:00<?, ?it/s]

Processed prompts: 100%|██████████| 20/20 [00:04<00:00,  4.26it/s, est. speed input: 62.44 toks/s, output: 482.23 toks/s]


In [9]:
gen_toks = sum(len(o.outputs[0].token_ids) for o in outs)
result = {
    "config": "fp16_t4",
    "throughput_tok_s": round(gen_toks / dt, 1),
    "peak_vram_gb": round(torch.cuda.max_memory_allocated() / 1e9, 2),
    "generations": [o.outputs[0].text for o in outs],   # reused in Day 8
}
json.dump(result, open("results_fp16_t4.json", "w"), indent=2)
print("tok/s", result["throughput_tok_s"], "| peak VRAM GB", result["peak_vram_gb"])

tok/s 476.0 | peak VRAM GB 0.0


The peak VRAM 0.0 is a measurement bug on the footprint script: torch.cuda.max_memory_allocated() only sees the parent process, but vLLM allocates in its EngineCore subprocess, so torch reports nothing. The throughput number comes back from the subprocess correctly; the VRAM probe doesn't.

For the VRAM number to be accurate, I will not use the torch call, instead I will write a python script and read the card with nvidia-smi inside the  `!python` process. In colab_fp16_benchmark.py, I will delete the torch.cuda.max_memory_allocated() line and replace the VRAM capture with:



```
def gpu_used_mib():
    out = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"])
    return int(out.decode().split("\n")[0])

# ... after llm.generate(...) finishes, instead of the torch line:
peak_vram_gb = gpu_used_mib() / 1024
```



In [2]:
!python colab_fp16_benchmark.py

INFO 09-15 10:34:08 [api_utils.py:286] non-default args: {'dtype': 'float16', 'max_model_len': 2048, 'disable_log_stats': True, 'model': 'Qwen/Qwen2.5-3B-Instruct'}
INFO 09-15 10:34:08 [model.py:684] Resolved architecture: Qwen2ForCausalLM
WARNING 09-15 10:34:08 [model.py:2355] Casting torch.bfloat16 to torch.float16.
INFO 09-15 10:34:08 [model.py:2021] Using max model len 2048
INFO 09-15 10:34:08 [scheduler.py:277] Chunked prefill is enabled with max_num_batched_tokens=8192.
Parse safetensors files: 100% 2/2 [00:00<00:00,  4.23it/s]
INFO 09-15 10:34:09 [kernel.py:369] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])
(EngineCore pid=14363) INFO 09-15 10:34:13 [core.py:123] Initializing a V1 LLM engine (v0.29.0) with config: model='Qwen/Qwen2.5-3B-Instruct', speculative_config=None, tokenizer='Qwen/Qwen2.5-3B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=main, tokenizer_revision=main, trust

### Result (Day 6)

FP16 batched throughput on the T4 was **479.7 tok/s** over my fixed prompt set (batched, not single-stream).

The VRAM numbers, kept straight:

- FP16 weights alone: **~5.8 GB** (from vLLM's `Model loading took …` log) — the number that matters, and why FP16 won't fit my 6 GB laptop.
- FP16 total card in use: **~13.2 GB** of the 16 GB T4 i.e. weights + the KV pool vLLM reserves up front (same reservation effect as Day 4).
- Local 4-bit weights: **~2.2 GB**, which is why 4-bit fits the 6 GB laptop.

FP16 weights ~5.8 GB (fills ~13.2 GB of a 16 GB T4 once serving) vs 4-bit weights ~2.2 GB on a 6 GB laptop. The tok/s across the two setups is illustrative, not a controlled benchmark (different GPUs, different batch conditions) — the apples-to-apples comparison is memory.